# Step 5 — The retrieval chain

Wire Gemini in so it answers **using the retrieved chunks**, and cites which
article each answer came from.

The original used `RetrievalQAWithSourcesChain`. In LangChain 1.x the entire
`langchain.chains` module **no longer exists** — so we build it with LCEL, the
current approach. It's also more transparent: you can see every stage.

> **Quota:** each question costs 1 embedding + 1 chat call. Chat is limited to
> **20/day** on the free tier, so don't loop this.

## Setup — load the saved index

In [ ]:
import os, warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

load_dotenv()
assert os.environ.get("GOOGLE_API_KEY"), "Put your key in rebuild/.env"

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2")
store = FAISS.load_local("faiss_index", embeddings,
                         allow_dangerous_deserialization=True)
print("loaded index with", store.index.ntotal, "vectors")

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", max_retries=3)

## 5.1 — The prompt

Two instructions do the heavy lifting:

1. **"using ONLY the passages"** — stops the model answering from its training
   data, which is the whole point of RAG
2. **"say so plainly"** if the answer isn't there — the alternative is a
   confident invention

The passages are numbered so the model can tell us which it used.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

class CitedAnswer(BaseModel):
    answer: str = Field(description="Answer using only the context. Say so if absent.")
    sources_used: list[int] = Field(description="Numbers of the passages actually used")

prompt = ChatPromptTemplate.from_template(
    "Answer the question using ONLY the numbered passages below.\n"
    "If the passages do not contain the answer, say so plainly.\n"
    "List the numbers of the passages you actually used.\n\n"
    "{context}\n\nQuestion: {question}"
)

# with_structured_output gives us a real object, not prose to parse.
# It also sidesteps Gemini 3.x returning .content as a list of blocks.
chain = prompt | llm.with_structured_output(CitedAnswer)
print("chain ready")

## 5.2 — First attempt, with k=4

`k` is how many chunks we hand the model.

In [ ]:
def format_docs(docs):
    """Number the passages so the model can cite them."""
    return "\n\n".join(f"[{i}] {d.page_content}" for i, d in enumerate(docs, 1))

def ask(question, k=4, show_chunks=False):
    retriever = store.as_retriever(search_kwargs={"k": k})
    docs = retriever.invoke(question)

    if show_chunks:
        print(f"retrieved {len(docs)} chunks:")
        for i, d in enumerate(docs, 1):
            print(f"  [{i}] {d.page_content[:70].replace(chr(10),' ')!r}")
        print()

    result = chain.invoke({"context": format_docs(docs), "question": question})

    print(f"Q: {question}")
    print(f"A: {result.answer}\n")
    urls = {docs[i-1].metadata["source"]
            for i in result.sources_used if 1 <= i <= len(docs)}
    print("Sources:")
    for u in sorted(urls) or ["(none cited)"]:
        print("  -", u)
    return result

_ = ask("Which company makes the Mahindra Thar?", k=4)

## ⚠️ That probably failed

You most likely got *"The provided passages do not contain the answer."*

The model is not broken — **it is being honest.** It answered correctly given
what it was shown. The failure happened one step earlier, in retrieval.

This is the single most common RAG bug: people blame the LLM when the retriever
never handed it the right text.

### Diagnose it: is the answer even in the index?

In [ ]:
all_docs = list(store.docstore._dict.values())
hits = [d for d in all_docs if "Thar" in d.page_content]
print(f"chunks containing 'Thar': {len(hits)} of {len(all_docs)}\n")

for d in hits[:2]:
    i = d.page_content.find("Thar")
    print("  ...", d.page_content[max(0, i-70):i+70].replace("\n", " "), "...\n")

The text **is** there. So why wasn't it retrieved? Check its rank:

In [ ]:
ranked = store.similarity_search("Which company makes the Mahindra Thar?",
                                 k=len(all_docs))
for rank, d in enumerate(ranked, 1):
    if "Thar" in d.page_content:
        print(f"first chunk mentioning 'Thar' is at rank {rank} of {len(all_docs)}")
        break

print("\nwith k=4 we only ever showed the model ranks 1-4.")

**Rank 6.** We asked for 4. The right passage was two places out of reach.

Why so low? Look at the top scores — they're all bunched together
(0.517, 0.536, 0.542...). Our chunks are noisy Wikipedia scrapes full of
citation markers and infobox fragments, so the embedding struggles to separate
"generally about Mahindra" from "specifically about the Thar".

## 5.3 — The fix

In [ ]:
_ = ask("Which company makes the Mahindra Thar?", k=8)

Now it answers and cites the source.

**Raising `k` is the cheap fix, not the best one.** More chunks means more
tokens per request, more cost, and more chance of confusing the model with
irrelevant text. Better long-term fixes:

- cleaner chunks (strip citation markers, tables, navigation)
- **MMR** retrieval, which favours diverse results over near-duplicates
- a reranker model that re-scores the top 50 candidates properly

## 5.4 — It must also refuse

A RAG system that answers everything is worse than useless — you can't tell
when it's guessing. Ask something the articles cannot possibly cover.

In [ ]:
_ = ask("What is the capital of France?", k=8)

It refuses, even though the model plainly *knows* the answer from training.
That's the `ONLY` instruction working, and it's what makes the tool
trustworthy.

## 5.5 — See the full pipeline

All the moving parts for one question:

In [ ]:
_ = ask("What vehicles does Tata Motors produce?", k=6, show_chunks=True)

---

## What just happened, end to end

1. Your question → embedded into a vector (1 API call)
2. FAISS finds the `k` nearest chunks
3. Chunks numbered `[1] [2] [3]...` and pasted into the prompt
4. Gemini answers **from those passages only** (1 API call)
5. It reports which numbers it used
6. We map those back to URLs

That's RAG complete.

## Checkpoint

**Next:** wrap this in a Streamlit UI — the original `main.py`, rebuilt.